# Coupled_Feedback3D — 3D one-way vs two-way coupling

3D twin of `MagneticCoupled/Coupled_Feedback`.


## Introduction

3D extrusion of the Lorentz hydrostatic balance case. The solution is z-independent; both the one-way control and the two-way discriminator are exercised in the mid-plane z=0.5.


In [ ]:
from trustutils import run
run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read TRUST probe (`.son`) file


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile
BUILD = run.BUILD_DIRECTORY
def read_son(case, name):
    cands = [f"{BUILD}/coarse/{case}_{name}.son", f"{BUILD}/{case}_{name}.son"]
    path = next((p for p in cands if os.path.exists(p)), cands[-1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)


## Run


In [ ]:
NB_PROCS = 2
run.reset()
case = run.addCase(".", "coupled_feedback_3d.data", nbProcs=NB_PROCS, excluNR=True)
case.partition()
case_twoway = run.addCase(".", "twoway_feedback_3d.data", nbProcs=NB_PROCS, excluNR=True)
case_twoway.partition()
run.printCases()
run.runCases()


## Checks

### #1 — one-way: b stays frozen


In [ ]:
t_bx, bx = read_son(case.dataFileName_, "SONDE_BX")
_, by = read_son(case.dataFileName_, "SONDE_BY")
bx_expected, by_expected = 0.35, -0.35
err_bx = float(np.max(np.abs(bx - bx_expected)))
err_by = float(np.max(np.abs(by - by_expected)))
print(f"max |bx - y|   = {err_bx:.3e}")
print(f"max |by - (-x)| = {err_by:.3e}")
assert err_bx < 1e-10 and err_by < 1e-10
print("OK: b stays frozen in one-way mode.")


### #2 — hydrostatic balance


In [ ]:
_, p1 = read_son(case.dataFileName_, "SONDE_P1")
_, p2 = read_son(case.dataFileName_, "SONDE_P2")
dp = float(p2[-1] - p1[-1])
expected = -(0.65**2 + 0.65**2 - 0.35**2 - 0.35**2)
err = abs(dp - expected) / abs(expected)
print(f"dp = {dp:.6g}   (analytic {expected:.4g}, err {err:.3e})")
assert dp < 0 and err < 0.05
print("OK: hydrostatic balance.")


### #3 — two-way discriminator


In [ ]:
CASE2 = case_twoway.dataFileName_
t_u, u = read_son(CASE2, "SONDE_U")
u_last = float(np.abs(u[-1]).max() if np.ndim(u)>0 else abs(u[-1]))
print(f"two-way: max |u| at t={t_u[-1]:.2f}: {u_last:.3e}")
assert u_last < 1e-3
_, p1 = read_son(CASE2, "SONDE_P1")
_, p2 = read_son(CASE2, "SONDE_P2")
dp = float(p2[-1] - p1[-1])
err = abs(dp - expected) / abs(expected)
print(f"two-way: dp = {dp:.6g}   (err {err:.3e})")
assert dp < 0 and err < 0.05
_, bx = read_son(CASE2, "SONDE_BX")
_, by = read_son(CASE2, "SONDE_BY")
dev = np.hypot(bx - 0.35, by + 0.35)
print(f"two-way: max deviation from frozen field = {dev.max():.4e}")
assert dev.max() > 1e-3 and dev.max() < 0.5
print("OK: two-way coupling is live.")


## Computer Performance


In [ ]:
run.tablePerf()